# Jeveration — source-faithful LiDeRe + Probe3D (95 images)

The exact published **LiDeRe** implicit spatial-attention/upscaling source (MIT) and exact **Probe3D** depth-bin / depth / normal loss source (MIT) are included in Jeveration. Three independently optimized readouts, unchanged 95 train / 20 validation Hypersim cache, and **1,200 optimizer updates each**. The previous approximate frozen-probe experiment remains untouched.

**Note:** This is a faithful port of the published readout and loss implementations to *Jeveration's cached frozen features*, not a reproduction of their paper's backbone or performance. Expect higher GPU costs than the old random-projection probe.

In [ ]:
# @title 1. Mount Google Drive, install the exact integrated research sources
from google.colab import drive
drive.mount("/content/drive")
import sys, subprocess, torch
SOURCE_COMMIT="035c9b440f47f41af80670a3d9f66e3169d4fd94" # pinned code
subprocess.run([sys.executable,"-m","pip","install","-q","--upgrade",
               f"git+https://github.com/Apache0ne/Jeveration.git@{SOURCE_COMMIT}"],check=True)
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Select BF16 L4 or A100"
print("GPU:",torch.cuda.get_device_name(0))


In [ ]:
# @title 2. Verify existing 95/20 cache and L4 settings
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/Jeveration_GT/pilot")
OUT=ROOT/"checkpoints_source_lidere"
LOCAL=Path("/content/jeveration_gt_pilot_cache")
assert len(list((ROOT/"cache/train").glob("*.npz")))==95
assert len(list((ROOT/"cache/val").glob("*.npz")))==20
OUT.mkdir(parents=True,exist_ok=True)
STEPS=1200 # @param {type:"integer"}
GRAD_ACCUM=16 # @param {type:"integer"}
LIDERE_DIM=64 # @param {type:"integer"}
LIDERE_BASE_SIZE=64 # @param {type:"integer"}
assert STEPS==1200 and LIDERE_DIM%8==0
print("Original 95 train, 20 validation, zero data changes")
print("New checkpoint directory:",OUT)


In [ ]:
# @title 3. Preflight + train with full underlying traceback (1,200 steps)
import sys, runpy, traceback
from pathlib import Path

def run_lidere_in_notebook(cli_args):
    old_argv=sys.argv[:]
    sys.argv=["jeveration.rgbd_source_lidere_train"]+list(cli_args)
    try:
        # In-process invocation preserves the original Python exception.
        # The older subprocess.run(check=True) hid it behind CalledProcessError.
        runpy.run_module("jeveration.rgbd_source_lidere_train",run_name="__main__",alter_sys=True)
    except BaseException:
        details=traceback.format_exc()
        print("\n====== ORIGINAL ERROR (not a subprocess wrapper) ======",flush=True)
        print(details[-16000:],flush=True)
        try:
            error_file=OUT/"last_lidere_error.txt"
            error_file.write_text(details,encoding="utf8")
            print("Traceback also saved:",error_file,flush=True)
        except Exception as saving_error:
            print("Could not save error log:",saving_error,flush=True)
        raise
    finally:
        sys.argv=old_argv

base=[
    "--root",str(ROOT),"--save-dir",str(OUT),
    "--local-cache-dir",str(LOCAL),"--device","cuda"]
preflight=[
    "preflight",*base,"--dim",str(LIDERE_DIM),
    "--base-size",str(LIDERE_BASE_SIZE),"--max-depth","80"]
print("Checking the REAL cached data and original-source forward/backward...",flush=True)
run_lidere_in_notebook(preflight)

cmd=[
    "fit",*base,"--cache-mode","ram",
    "--steps",str(STEPS),"--batch-size","1",
    "--grad-accum",str(GRAD_ACCUM),"--dim",str(LIDERE_DIM),
    "--base-size",str(LIDERE_BASE_SIZE),"--max-depth","80",
    "--lr-depth","0.00015","--lr-normal","0.00015","--lr-seg","0.00015",
    "--eval-every","100","--save-every","200","--save-at","400,700,900"]
if (OUT/"latest_source.json").exists():cmd.append("--resume")
run_lidere_in_notebook(cmd)


In [ ]:
# @title 4. Export the same 20-image holdout report and copy to Drive
import shutil, json
from pathlib import Path
from google.colab import files
run_lidere_in_notebook([
    "report","--root",str(ROOT),"--save-dir",str(OUT),
    "--local-cache-dir",str(LOCAL),"--device","cuda",
    "--output","/content/Jeveration_Source_LiDeRe_Report"
])
report=Path("/content/Jeveration_Source_LiDeRe_Probe3D_Report.zip")
assert report.is_file()
dest=ROOT/"reports"/report.name
dest.parent.mkdir(parents=True,exist_ok=True)
shutil.copy2(report,dest)
metrics=json.loads(Path("/content/Jeveration_Source_LiDeRe_Report/metrics.json").read_text())
print(json.dumps(metrics,indent=2))
print("Report saved:",dest)
files.download(str(report))
